# The Price Is Right — Unsloth QLoRA Fine-Tuning + Evaluation
### Qwen2.5-Math-1.5B · ed-donner/items_prompts_full · H100 80 GB

**Why Unsloth over vanilla HuggingFace?**
- **2× faster inference** via custom CUDA kernels (`FastLanguageModel.for_inference`)
- **60–80% less VRAM** during training via Unsloth's optimised backward pass
- **`packing=True`** in SFTTrainer: packs multiple short items per batch — critical H100 throughput win
- **`lora_dropout=0`**: Unsloth-recommended; empirically improves convergence over dropout > 0
- **`use_gradient_checkpointing="unsloth"`**: 30% longer context at same VRAM vs standard checkpointing
- **`adamw_8bit`**: 8-bit optimizer states instead of 32-bit — saves ~2 GB on 1.5B model

**Notebook flow**
1. Environment setup (cache redirect, install, imports)
2. Constants & hyperparameters
3. HuggingFace login
4. Load & inspect dataset
5. Load base model with Unsloth
6. Base model evaluation — diverse beam search + geometric mean
7. Prepare dataset for SFT (formatting function)
8. QLoRA fine-tuning with Unsloth SFTTrainer
9. Push final model to Hub
10. Load fine-tuned model for inference
11. Fine-tuned model evaluation
12. Side-by-side comparison & results


## 1 · Environment Setup

In [22]:
import os

# ── Redirect HF cache away from quota-limited home directory ─────────────────
# /tmp is the scratch partition on this H100 node
os.environ["HF_HOME"]            = "/tmp/mawojide/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/mawojide/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"]  = "/tmp/mawojide/hf_cache/datasets"

os.makedirs("/tmp/mawojide/hf_cache/transformers", exist_ok=True)
os.makedirs("/tmp/mawojide/hf_cache/datasets",     exist_ok=True)

print("HF_HOME →", os.environ["HF_HOME"])


HF_HOME → /tmp/mawojide/hf_cache


In [23]:
# Install Unsloth + dependencies
# Unsloth auto-detects CUDA version and installs matching wheels
# ! pip install "unsloth[cu121-ampere-torch230]" -q   # CUDA 12.1, Ampere/H100
# ! pip install unsloth -q                             # generic fallback
# ! pip install scikit-learn plotly -q                 # evaluation deps


In [24]:
! python -m pip install wandb weave

In [25]:
import os, re, math, statistics
from datetime import datetime
from itertools import accumulate
from IPython.display import clear_output

import torch
from unsloth import FastLanguageModel           # replaces AutoModelForCausalLM
from trl import SFTTrainer
from transformers import TrainingArguments, set_seed
from datasets import load_dataset
from huggingface_hub import login

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

print("torch         :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU           :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
capability = torch.cuda.get_device_capability()
USE_BF16   = capability[0] >= 8   # True on H100 / A100
print(f"bfloat16      : {USE_BF16}  (compute capability {capability})")


torch         : 2.10.0+cu128
CUDA available: True
GPU           : NVIDIA H100 80GB HBM3
bfloat16      : True  (compute capability (9, 0))


## 2 · Constants & Hyperparameters

In [26]:
# ── Model ─────────────────────────────────────────────────────────────────────
# unsloth/ prefix gives Unsloth's pre-patched, flash-attention-ready weights
# bnb-4bit skips runtime quantization → faster to load
BASE_MODEL   = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit"   # Best base: 86.7 MAE
# BASE_MODEL = "Qwen/Qwen2.5-Math-1.5B"
PROJECT_NAME = "price"
HF_USER      = "martinsawojide"

# ── Dataset ───────────────────────────────────────────────────────────────────
LITE_MODE    = False    # False → full dataset (recommended on H100)
DATA_USER    = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# ── Run naming ────────────────────────────────────────────────────────────────
RUN_NAME         = f"{datetime.now():%Y-%m-%d_%H.%M.%S}" + ("-lite" if LITE_MODE else "")
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME   = f"{HF_USER}/{PROJECT_RUN_NAME}"

# ── Sequence length ───────────────────────────────────────────────────────────
# Must be set at model-load time with Unsloth (not in TrainingArguments)
# Our prompts are short (~80-100 tokens); 256 gives headroom + enables packing
MAX_SEQ_LENGTH = 256

# ── QLoRA (Unsloth recommendations) ──────────────────────────────────────────
# r=64 full run, r=16 lite run — Unsloth handles alpha internally if not set
LORA_R       = 16 if LITE_MODE else 64
LORA_ALPHA   = LORA_R * 2      # standard 2× rule
LORA_DROPOUT = 0               # Unsloth explicitly recommends 0; no degradation empirically
# Full attention + MLP targeting for regression precision (lite: attention only)
ATTENTION_LAYERS = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP_LAYERS       = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES   = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS          = 1   if LITE_MODE else 3
# H100 + 1.5B + packing=True: 32 per device is conservative — push to 64 if VRAM allows
BATCH_SIZE      = 32  if LITE_MODE else 64
GRAD_ACCUM      = 1                    # effective batch = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE   = 1e-4                 # validated sweet spot from literature for 1.5B SFT
WARMUP_STEPS    = 50  if LITE_MODE else 100
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.05                 # slightly higher than v2 — prevents outlier overfitting
# adamw_8bit: 8-bit optimizer states, saves ~2 GB vs paged_adamw_32bit
OPTIMIZER       = "adamw_8bit"
MAX_GRAD_NORM   = 0.3

# ── Dataset ──────────────────────────────────────────────────────────
dataset   = load_dataset(DATASET_NAME)
train     = dataset["train"]
val       = dataset["val"]  
test      = dataset["test"]

print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")

# ── Logging / saving ──────────────────────────────────────────────────────────
LOG_STEPS    = 5    if LITE_MODE else 25
SAVE_STEPS   = 100  if LITE_MODE else 250
VAL_SIZE     = 500  if LITE_MODE else len(val)
LOG_TO_WANDB = True   # set True + set WANDB_API_KEY env var to enable

# ── Inference (best config validated through experiments) ─────────────────────
BEAM_NUM_BEAMS   = 10
BEAM_NUM_GROUPS  = 10
BEAM_DIV_PENALTY = 1.5
EVAL_SIZE        = len(test)

print(f"Run name       : {RUN_NAME}")
print(f"Hub model      : {HUB_MODEL_NAME}")
print(f"Dataset        : {DATASET_NAME}")
print(f"Max seq length : {MAX_SEQ_LENGTH}")
print(f"LoRA r / alpha : {LORA_R} / {LORA_ALPHA}")
print(f"Batch / epochs : {BATCH_SIZE} / {EPOCHS}")
print(f"Optimizer      : {OPTIMIZER}")


Train : 800,000
Val   : 10,000
Test  : 10,000
Run name       : 2026-03-06_11.54.32
Hub model      : martinsawojide/price-2026-03-06_11.54.32
Dataset        : ed-donner/items_prompts_full
Max seq length : 256
LoRA r / alpha : 64 / 128
Batch / epochs : 64 / 3
Optimizer      : adamw_8bit


## 3 · HuggingFace Login

In [27]:
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)


In [31]:
# ! python -m pip uninstall wandb -y
# ! python -m pip install wandb

In [32]:
# ── Optional: Weights & Biases ────────────────────────────────────────────────
import wandb
os.environ["WANDB_API_KEY"]   = os.getenv("WANDB_API_KEY")
os.environ["WANDB_PROJECT"]   = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"]     = "false"
wandb.login()


ImportError: cannot import name 'Imports' from 'wandb.proto.wandb_telemetry_pb2' (/home/mawojide/orchard_project_trial/orchard_venv_3_11/lib/python3.11/site-packages/wandb/proto/wandb_telemetry_pb2.py)

## 4 · Load & Inspect Dataset

In [ ]:
# dataset = load_dataset(DATASET_NAME)
# train   = dataset["train"]
# val     = dataset["val"].select(range(VAL_SIZE))
# test    = dataset["test"]

# print(f"Train : {len(train):,} rows")
# print(f"Val   : {len(val):,} rows")
# print(f"Test  : {len(test):,} rows")
# print(f"\nColumns : {train.column_names}")
# print(f"\nSample prompt (first 300 chars):\n{train[0]['prompt'][:300]}")
# print(f"\nSample completion : {train[0]['completion']}")


## 5 · Load Base Model with Unsloth

`FastLanguageModel.from_pretrained` replaces the verbose HuggingFace stack:
- No manual `BitsAndBytesConfig` — `load_in_4bit=True` handles it
- `max_seq_length` set here once, not scattered across configs
- `dtype=torch.bfloat16` maps directly to H100 native precision
- Model arrives with Unsloth's kernel patches already applied


In [ ]:
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,   # H100 native — no performance loss vs float32
    load_in_4bit   = True,             # NF4 quantization
    # token        = hf_token,         # uncomment for gated models
)

# Unsloth requires pad_token to be set for generation
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
base_model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded  : {BASE_MODEL}")
print(f"Memory        : {base_model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Pad token id  : {tokenizer.pad_token_id}")


## 6 · Inference Helpers

**Important Unsloth pattern:**
- Training mode  → model as returned by `FastLanguageModel.from_pretrained`
- Inference mode → call `FastLanguageModel.for_inference(model)` first

`for_inference` enables Unsloth's 2× faster attention kernels.  
It is called automatically in the evaluation helpers below.


In [ ]:
def extract_price(text: str):
    """Extract the first plausible USD price from generated text."""
    patterns = [
        r"\$[\d,]+\.?\d*",   # $12.99  $1,299
        r"[\d,]+\.\d{2}",    # 12.99
        r"[\d,]+",             # 12
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            price_str = match.group().replace("$", "").replace(",", "")
            try:
                price = float(price_str)
                if 0.01 <= price <= 100_000:
                    return price
            except ValueError:
                continue
    return None


def predict_with_model(model, item,
                       num_beams=BEAM_NUM_BEAMS,
                       num_groups=BEAM_NUM_GROUPS,
                       diversity_penalty=BEAM_DIV_PENALTY):
    """
    Diverse beam search → geometric mean of all beam prices.
    Best config found: 10 beams / 5 groups / penalty=1.0 → 86.7 MAE on base model.
    Works with both base model and fine-tuned Unsloth model.
    """
    # Switch to Unsloth's fast inference kernels
    FastLanguageModel.for_inference(model)

    prompt       = str(item["prompt"])
    inputs       = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens      = 8,
            num_beams           = num_beams,
            num_beam_groups     = num_groups,
            diversity_penalty   = diversity_penalty,
            num_return_sequences= num_beams,
            pad_token_id        = tokenizer.eos_token_id,
            use_cache           = True,    # Unsloth requires this for KV cache speedup
        )

    prices = []
    for beam_output in output_ids:
        text  = tokenizer.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    if not prices:
        return None
    return statistics.geometric_mean(prices)


# Named wrapper so Tester.make_title() generates a clean label
def base_model_predict(item):
    return predict_with_model(base_model, item)


## 7 · Evaluation Framework (Tester + Plotly Charts)

In [ ]:
GREEN     = "\033[92m"
YELLOW    = "\033[93m"
RED       = "\033[91m"
RESET     = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


class Tester:
    def __init__(self, predictor, data, title=None, size=EVAL_SIZE):
        self.predictor = predictor
        self.data      = data
        self.title     = title or self.make_title(predictor)
        self.size      = size
        self.titles    = []
        self.guesses   = []
        self.truths    = []
        self.errors    = []
        self.colors    = []

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title().replace("Gpt", "GPT")

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        return value if value is not None else 0

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:    return "green"
        elif error < 80 or error / truth < 0.4:  return "orange"
        else:                                     return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value     = self.predictor(datapoint)
        guess     = self.post_process(value)
        truth     = float(datapoint["completion"])
        error     = abs(guess - truth)
        color     = self.color_for(error, truth)
        pieces    = datapoint["prompt"].split("Title: ")
        title     = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0]
        title     = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({
            "truth": self.truths, "guess": self.guesses,
            "title": self.titles, "error": self.errors, "color": self.colors,
        })
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title, labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=800, height=600,
        )
        for tr in fig.data:
            mask         = df["color"] == tr.name
            tr.customdata   = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            hoverinfo="skip", showlegend=False,
        ))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n               = len(self.errors)
        running_sums    = list(accumulate(self.errors))
        x               = list(range(1, n + 1))
        running_means   = [s / i for s, i in zip(running_sums, x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds    = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]
        ci    = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        title = f"{self.title}  |  Final MAE: ${running_means[-1]:,.2f} ± ${ci[-1]:,.2f}"
        fig   = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=x, y=running_means, mode="lines",
            line=dict(width=3, color="firebrick"),
            customdata=list(zip(ci)),
            hovertemplate="n=%{x}<br>Avg Error=$%{y:,.2f}<br>±95% CI=$%{customdata[0]:,.2f}<extra></extra>",
        ))
        fig.update_layout(
            title=title, xaxis_title="Number of Datapoints",
            yaxis_title="MAE ($)", width=800, height=300,
            template="plotly_white", showlegend=False,
        )
        fig.show()

    def report(self):
        avg_error = sum(self.errors) / self.size
        mse       = mean_squared_error(self.truths, self.guesses)
        r2        = r2_score(self.truths, self.guesses) * 100
        title     = (f"{self.title}<br>"
                     f"<b>MAE:</b> ${avg_error:,.2f}  "
                     f"<b>MSE:</b> {mse:,.0f}  "
                     f"<b>r²:</b> {r2:.1f}%")
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size), desc=self.title):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        clear_output(wait=True)
        self.report()
        return self  # allow chaining


def evaluate(function, data, size=EVAL_SIZE):
    return Tester(function, data, size=size).run()


## 8 · Base Model Evaluation

Benchmarks the unmodified Qwen2.5-Math-1.5B with our best inference recipe:
**diverse beam search (10/5/1.0) + geometric mean** → **86.7 MAE** (established in prior experiments).


In [ ]:
# Smoke test — single item before committing to full eval
sample    = test[0]
pred      = base_model_predict(sample)
truth     = float(sample["completion"])
print(f"Prompt (tail) : ...{sample['prompt'][-80:]}")
print(f"Prediction    : ${pred:.2f}")
print(f"Ground truth  : ${truth:.2f}")
print(f"Error         : ${abs(pred - truth):.2f}")


In [ ]:
set_seed(42)
base_tester = evaluate(base_model_predict, test, size=EVAL_SIZE)


## 9 · Prepare Dataset for SFT

Unsloth's `SFTTrainer` expects a single `text` field containing the **full sequence**
(prompt + completion + EOS token). We create this with a mapping function.

**Why append EOS?** It trains the model to produce a clean stop after the price,
preventing runaway generation at inference time.

**Why `packing=True`?** Our prompts are ~80-120 tokens. Without packing, most of
each 256-token context window is wasted padding. Packing fills windows completely,
multiplying effective throughput on the H100 by 2-3×.


In [ ]:
def format_for_sft(examples):
    """
    Combine prompt + completion into a single text field.
    completion is the price string e.g. '12.99' — append EOS so model learns to stop.
    """
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        # Full sequence: prompt already ends with 'Price is $', completion is the number
        full_text = str(prompt) + str(completion) + tokenizer.eos_token
        texts.append(full_text)
    return {"text": texts}


# Map in batches — fast, avoids Python loop overhead
train_sft = train.map(format_for_sft, batched=True, remove_columns=train.column_names)
val_sft   = val.map(format_for_sft,   batched=True, remove_columns=val.column_names)

# Verify
print("Sample SFT text (first 200 chars):")
print(train_sft[0]["text"][:200])
print(f"\nTrain SFT size : {len(train_sft):,}")
print(f"Val SFT size   : {len(val_sft):,}")


## 10 · Add LoRA Adapters with Unsloth

`FastLanguageModel.get_peft_model` replaces the manual `LoraConfig` + `get_peft_model` flow.

Key differences from HuggingFace approach:
- **`lora_dropout=0`** — Unsloth's explicit recommendation; better final MAE empirically
- **`use_gradient_checkpointing="unsloth"`** — their custom implementation allows 30% longer
  sequences at same VRAM vs `True`
- **`random_state=3407`** — Unsloth's recommended seed for reproducibility


In [ ]:
# Switch base_model from inference mode → training mode
# (FastLanguageModel.for_inference was called during eval above)
base_model.train()

base_model = FastLanguageModel.get_peft_model(
    base_model,
    r                   = LORA_R,
    target_modules      = TARGET_MODULES,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,   # 0 — Unsloth recommendation
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # 30% more context, same VRAM
    random_state        = 3407,
    use_rslora          = True,          # rank-stabilised LoRA — set True for r >= 128
    loftq_config        = None,
)

# Count trainable parameters
trainable   = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total       = sum(p.numel() for p in base_model.parameters())
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of total)")
print(f"Total params     : {total:,}")
print(f"LoRA rank        : {LORA_R}  |  alpha: {LORA_ALPHA}  |  dropout: {LORA_DROPOUT}")
print(f"Target modules   : {TARGET_MODULES}")


## 11 · Fine-Tune with Unsloth SFTTrainer

### Key H100 optimizations active:
- `packing=True` — fills context windows, 2-3× throughput vs padded batches
- `bf16=True` — H100 native precision, no precision-loss vs fp32
- `adamw_8bit` — 8-bit optimizer states save ~2 GB VRAM
- `use_gradient_checkpointing="unsloth"` — already set in LoRA config above


In [ ]:
train_args = TrainingArguments(
    # ── Output ───────────────────────────────────────────────────────────────
    output_dir          = PROJECT_RUN_NAME,

    # ── Training duration ────────────────────────────────────────────────────
    num_train_epochs    = EPOCHS,
    max_steps           = -1,           # -1 = run full epochs

    # ── Batch & gradient ─────────────────────────────────────────────────────
    per_device_train_batch_size  = BATCH_SIZE,
    per_device_eval_batch_size   = 1,
    gradient_accumulation_steps  = GRAD_ACCUM,

    # ── Optimiser & LR ───────────────────────────────────────────────────────
    optim               = OPTIMIZER,    # adamw_8bit
    learning_rate       = LEARNING_RATE,
    weight_decay        = WEIGHT_DECAY,
    warmup_steps        = WARMUP_STEPS,
    lr_scheduler_type   = LR_SCHEDULER,
    max_grad_norm       = MAX_GRAD_NORM,

    # ── Precision (H100) ─────────────────────────────────────────────────────
    fp16                = False,
    bf16                = USE_BF16,     # True on H100

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps       = LOG_STEPS,
    report_to           = "wandb" if LOG_TO_WANDB else "none",
    run_name            = RUN_NAME,

    # ── Checkpointing ────────────────────────────────────────────────────────
    save_strategy       = "steps",
    save_steps          = SAVE_STEPS,
    save_total_limit    = 5,            # keep last 5 checkpoints to save disk

    # ── Evaluation ───────────────────────────────────────────────────────────
    eval_strategy       = "steps",
    eval_steps          = SAVE_STEPS,

    # ── Hub push ─────────────────────────────────────────────────────────────
    push_to_hub         = True,
    hub_model_id        = HUB_MODEL_NAME,
    hub_strategy        = "every_save",
    hub_private_repo    = True,

    # ── Misc ─────────────────────────────────────────────────────────────────
    seed                = 3407,
    dataloader_num_workers = 4,        # H100 has CPU headroom to parallelise data loading
)

print(f"Output dir     : {PROJECT_RUN_NAME}")
print(f"Hub target     : {HUB_MODEL_NAME}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
trainer = SFTTrainer(
    model           = base_model,
    tokenizer       = tokenizer,
    train_dataset   = train_sft,
    eval_dataset    = val_sft,
    dataset_text_field = "text",        # the column we created in format_for_sft
    max_seq_length  = MAX_SEQ_LENGTH,
    packing         = True,             # ← critical H100 throughput win
    args            = train_args,
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"Trainer ready — {trainable:,} trainable parameters")


### Start training

Checkpoints are pushed to HuggingFace Hub every `SAVE_STEPS` steps.  
If the session is interrupted, reload the latest checkpoint:
```python
trainer.train(resume_from_checkpoint=True)
```


In [ ]:
if LOG_TO_WANDB:
    import wandb
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps    : {trainer_stats.global_step}")
print(f"Training loss  : {trainer_stats.training_loss:.4f}")
print(f"Runtime        : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")

if LOG_TO_WANDB:
    wandb.finish()


## 12 · Save & Push Final Model

In [ ]:
# Save LoRA adapter weights only (not the full model — much smaller)
trainer.model.save_pretrained(PROJECT_RUN_NAME)
tokenizer.save_pretrained(PROJECT_RUN_NAME)
print(f"Saved locally  → {PROJECT_RUN_NAME}/")

# Push final adapter to Hub
trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Pushed to Hub  → {HUB_MODEL_NAME}")


In [ ]:
# ── Optional: save full merged model (base + adapter merged into one) ─────────
# Merging gives a standalone model that doesn't need the adapter at load time
# Costs more disk but simplifies deployment

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(f"{PROJECT_RUN_NAME}-merged", safe_serialization=True)
tokenizer.save_pretrained(f"{PROJECT_RUN_NAME}-merged")
print(f"Merged model saved → {PROJECT_RUN_NAME}-merged/")


## 13 · Load Fine-Tuned Model for Inference

`FastLanguageModel.from_pretrained` with the Hub adapter ID loads both the base weights
and the LoRA adapter in one call — no separate `PeftModel.from_pretrained` needed.

Then `FastLanguageModel.for_inference` activates the 2× speed kernels.


In [ ]:
# ── If loading a previously saved run instead of the one just trained: ─────────
# HUB_MODEL_NAME = "martinsawojide/price-YYYY-MM-DD_HH.MM.SS"

fine_tuned_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = HUB_MODEL_NAME,   # adapter + base weights loaded together
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = True,
)

# Activate Unsloth's fast inference kernels — required before generation
FastLanguageModel.for_inference(fine_tuned_model)

# Ensure pad token matches training setup
ft_tokenizer.pad_token    = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"
fine_tuned_model.config.pad_token_id = ft_tokenizer.eos_token_id

print(f"Fine-tuned model loaded : {HUB_MODEL_NAME}")
print(f"Memory footprint        : {fine_tuned_model.get_memory_footprint() / 1e9:.2f} GB")


## 14 · Fine-Tuned Model Evaluation

Same inference recipe as the base model benchmark for a fair apples-to-apples comparison:
**diverse beam search (10/5/1.0) + geometric mean**


In [ ]:
def fine_tuned_predict(item):
    """Wrapper using fine-tuned model — for_inference already called above."""
    prompt       = str(item["prompt"])
    inputs       = ft_tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(
            **inputs,
            max_new_tokens       = 8,
            num_beams            = BEAM_NUM_BEAMS,
            num_beam_groups      = BEAM_NUM_GROUPS,
            diversity_penalty    = BEAM_DIV_PENALTY,
            num_return_sequences = BEAM_NUM_BEAMS,
            pad_token_id         = ft_tokenizer.eos_token_id,
            use_cache            = True,
        )

    prices = []
    for beam_output in output_ids:
        text  = ft_tokenizer.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    return statistics.geometric_mean(prices) if prices else None


In [ ]:
# Smoke test — compare single item
sample     = test[0]
pred_base  = base_model_predict(sample)
pred_ft    = fine_tuned_predict(sample)
truth      = float(sample["completion"])

print(f"Prompt tail      : ...{sample['prompt'][-80:]}")
print(f"Base prediction  : ${pred_base:.2f}  (error ${abs(pred_base - truth):.2f})")
print(f"FT prediction    : ${pred_ft:.2f}  (error ${abs(pred_ft - truth):.2f})")
print(f"Ground truth     : ${truth:.2f}")


In [ ]:
set_seed(42)
ft_tester = evaluate(fine_tuned_predict, test, size=EVAL_SIZE)


## 15 · Side-by-Side Comparison & Results

In [ ]:
# Re-run base tester if not already stored from Section 8
# base_tester is already defined — reuse it
base_mae = sum(base_tester.errors) / EVAL_SIZE
ft_mae   = sum(ft_tester.errors)   / EVAL_SIZE
delta    = base_mae - ft_mae

base_mse = mean_squared_error(base_tester.truths, base_tester.guesses)
ft_mse   = mean_squared_error(ft_tester.truths,   ft_tester.guesses)
base_r2  = r2_score(base_tester.truths, base_tester.guesses) * 100
ft_r2    = r2_score(ft_tester.truths,   ft_tester.guesses)   * 100

print(f"{'Metric':<12} {'Base Model':>14} {'Fine-Tuned':>14} {'Improvement':>14}")
print("─" * 58)
print(f"{'MAE ($)':<12} {base_mae:>14.2f} {ft_mae:>14.2f} {delta:>+14.2f}")
print(f"{'MSE':<12} {base_mse:>14.0f} {ft_mse:>14.0f} {base_mse - ft_mse:>+14.0f}")
print(f"{'r² (%)':<12} {base_r2:>14.1f} {ft_r2:>14.1f} {ft_r2 - base_r2:>+14.1f}")
print("─" * 58)
print(f"\n→ Fine-tuning improved MAE by ${delta:.2f} ({delta / base_mae * 100:.1f}%)")


In [ ]:
# Overlay scatter: base vs fine-tuned on same axes
fig = go.Figure()

for tester, name, color in [
    (base_tester, "Base Model", "royalblue"),
    (ft_tester,   "Fine-Tuned", "crimson"),
]:
    fig.add_trace(go.Scatter(
        x=tester.truths, y=tester.guesses,
        mode="markers", name=name,
        marker=dict(color=color, size=5, opacity=0.6),
    ))

max_val = max(max(base_tester.truths), max(ft_tester.truths),
              max(base_tester.guesses), max(ft_tester.guesses))
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val], mode="lines",
    line=dict(dash="dash", color="gray", width=1),
    name="Perfect prediction", hoverinfo="skip",
))

fig.update_layout(
    title=f"Base vs Fine-Tuned | MAE: Base ${base_mae:.2f} → FT ${ft_mae:.2f}",
    xaxis_title="Actual Price ($)",
    yaxis_title="Predicted Price ($)",
    width=900, height=600,
    template="plotly_white",
)
fig.update_xaxes(range=[0, max_val])
fig.update_yaxes(range=[0, max_val])
fig.show()
